> **Note:** originally developed and run in Google Colab. Google-Drive mounting cells were removed, Drive paths were rewritten as relative paths and outputs were stripped (figures of the original run are in `assets/`). The paths/cleanup were **not re-executed** after this change, so expect to adjust paths.

#**Cargar modelo**


In [ ]:
!pip install albumentations==1.4.3


import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.measure import label, regionprops
from skimage.metrics import hausdorff_distance

import matplotlib.pyplot as plt
from tqdm import tqdm


base_path = "."
pairs_csv = Path(base_path) / "metadata/train_pairs.csv"

df_pairs = pd.read_csv(pairs_csv)
print(df_pairs.head())
print("Total pares:", len(df_pairs))

In [ ]:
models_dir = Path(base_path) / "models"
best_model_path = models_dir / "best_unet_glas.pth"

print(best_model_path, best_model_path.exists())


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.down1 = DoubleConv(in_ch, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)
        c2 = self.down2(p1)
        p2 = self.pool(c2)
        c3 = self.down3(p2)
        p3 = self.pool(c3)
        c4 = self.down4(p3)
        p4 = self.pool(c4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn)
        u4 = torch.cat([u4, c4], dim=1)
        c4 = self.conv4(u4)
        u3 = self.up3(c4)
        u3 = torch.cat([u3, c3], dim=1)
        c3 = self.conv3(u3)
        u2 = self.up2(c3)
        u2 = torch.cat([u2, c2], dim=1)
        c2 = self.conv2(u2)
        u1 = self.up1(c2)
        u1 = torch.cat([u1, c1], dim=1)
        c1 = self.conv1(u1)

        return self.final(c1)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = UNet().to(device)

state_dict = torch.load(best_model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("Modelo cargado correctamente.")

In [ ]:
val_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

In [ ]:
best_thr = 0.45  # o el valor que te haya dado antes
print("best_thr:", best_thr)


In [ ]:
from skimage.morphology import remove_small_objects, binary_opening, disk

def clean_mask(pred_bin, min_size=100, opening_radius=2):
    """
    pred_bin: máscara binaria [H,W] (0/1)
    Devuelve máscara binaria limpia [H,W].
    """
    mask = pred_bin.astype(bool)

    mask = remove_small_objects(mask, min_size=min_size)

    if opening_radius > 0:
        selem = disk(opening_radius)
        mask = binary_opening(mask, selem)

    return mask.astype(np.uint8)

In [ ]:
test_images_dir = Path(base_path) / "data/test/images"
example_path = sorted(test_images_dir.glob("*.bmp"))[0]
print("Ejemplo:", example_path)

image = np.array(Image.open(example_path).convert("RGB"))

aug = val_transforms(image=image)
img_tensor = aug["image"].unsqueeze(0).to(device)  # [1,3,H,W]

with torch.no_grad():
    logits = model(img_tensor)
    prob = torch.sigmoid(logits)[0,0].cpu().numpy()  # [H,W]

pred_bin_raw = (prob > best_thr).astype(np.uint8)
pred_bin = clean_mask(pred_bin_raw)

plt.figure(figsize=(10,4))
plt.subplot(1,3,1); plt.imshow(image); plt.title("Imagen original"); plt.axis("off")
plt.subplot(1,3,2); plt.imshow(pred_bin_raw, cmap="gray"); plt.title("Pred cruda"); plt.axis("off")
plt.subplot(1,3,3); plt.imshow(pred_bin, cmap="gray"); plt.title("Pred limpia"); plt.axis("off")
plt.tight_layout()
plt.show()

# **Deep features**

In [ ]:
# ============================================================
# 1) GENERAR CROPS GLANDULARES + DF CON FEATURES
# ============================================================
from skimage.measure import label, regionprops

crop_dir = Path(base_path) / "data/gland_crops"
crop_dir.mkdir(parents=True, exist_ok=True)

gland_dfs = []

for path in tqdm(all_test_paths):
    # A = benigno, B = maligno (igual que antes)
    tag = "A" if "testA" in path.name else "B"
    image = np.array(Image.open(path).convert("RGB"))

    # Segmentación (igual que antes)
    aug = val_transforms(image=image)
    img_tensor = aug["image"].unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img_tensor)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

    pred_bin_raw = (prob > best_thr).astype(np.uint8)
    pred_bin = clean_mask(pred_bin_raw)

    # Extraer features por glándula (tu función)
    df_img = extract_gland_features(pred_bin, tag, image_rgb=image)

    # Volvemos a etiquetar para alinear props con filas de df_img
    labeled = label(pred_bin)
    props = regionprops(labeled)

    # Redimensionar imagen al tamaño de la máscara (igual que en extract_gland_features)
    Hm, Wm = pred_bin.shape
    img_resized = cv2.resize(image, (Wm, Hm), interpolation=cv2.INTER_LINEAR)

    crop_paths = []
    gland_ids = []

    for idx, p in enumerate(props):
        minr, minc, maxr, maxc = p.bbox

        # padding alrededor de la glándula
        pad = 10
        minr_p = max(minr - pad, 0)
        minc_p = max(minc - pad, 0)
        maxr_p = min(maxr + pad, Hm)
        maxc_p = min(maxc + pad, Wm)

        crop_rgb = img_resized[minr_p:maxr_p, minc_p:maxc_p]

        crop_path = crop_dir / f"{path.stem}_g{idx:03d}.png"
        Image.fromarray(crop_rgb).save(crop_path)

        crop_paths.append(str(crop_path))
        gland_ids.append(idx)

    # Asegurarnos de que coincida el número de glándulas
    if len(df_img) != len(crop_paths):
        print(f"[WARN] Diferencia en #glándulas para {path.name}: df={len(df_img)} props={len(crop_paths)}")

    df_img = df_img.iloc[:len(crop_paths)].copy()
    df_img["crop_path"] = crop_paths[:len(df_img)]
    df_img["gland_id"] = gland_ids[:len(df_img)]
    df_img["image_name"] = path.name

    gland_dfs.append(df_img)

df_glands = pd.concat(gland_dfs, ignore_index=True)
print("df_glands.shape:", df_glands.shape)
df_glands.head()

In [ ]:
# ============================================================
# 2) DATASET Pytorch: IMAGEN + FEATURES
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# mismas columnas que antes, pero usando df_glands
feature_cols = [
    "area", "perimeter", "circularity", "solidity", "eccentricity", "irregularity",
    "lumen_ratio", "border_roughness", "nn_distance",
    "fractal_dim", "skeleton_length", "skeleton_endpoints", "skeleton_branchpoints",
    "contour_mean_curv", "contour_std_curv",
    "haralick_contrast", "haralick_homogeneity", "haralick_energy", "haralick_correlation",
    "gabor_mean", "gabor_std",
    "H_mean", "H_std", "E_mean", "E_std"
]
feature_cols = [c for c in feature_cols if c in df_glands.columns]

# transforms para EfficientNet-B0 (224x224 + normalización ImageNet)
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

class GlandDataset(Dataset):
    def __init__(self, df, feature_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["crop_path"]).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        feats = torch.tensor(row[self.feature_cols].values, dtype=torch.float32)
        label = torch.tensor(row["label"], dtype=torch.float32)

        return img, feats, label


In [ ]:
# ============================================================
# 3) MODELO: EFFICIENTNET-B0 + FEATURES
# ============================================================
!pip install -q timm

import timm
import torch.nn as nn
import torch.nn.functional as F

class GlandNetEffB0(nn.Module):
    def __init__(self, n_features, fuse_morph=True, pretrained=True):
        super().__init__()
        self.fuse_morph = fuse_morph

        # backbone sin cabeza final, con global pooling
        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=pretrained,
            num_classes=0,      # sin clasificación
            global_pool="avg"   # vector 1280
        )
        backbone_dim = self.backbone.num_features  # normalmente 1280

        in_dim = backbone_dim + (n_features if fuse_morph else 0)

        self.classifier = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1)   # salida logit
        )

    def forward(self, x_img, x_feats=None):
        emb = self.backbone(x_img)  # [B, backbone_dim]

        if self.fuse_morph:
            if x_feats is None:
                raise ValueError("fuse_morph=True pero x_feats es None")
            emb = torch.cat([emb, x_feats], dim=1)

        logits = self.classifier(emb).squeeze(1)
        return logits


In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df_glands,
    test_size=0.25,
    random_state=42,
    stratify=df_glands["label"]
)

print("Train glands:", df_train.shape[0], "Val glands:", df_val.shape[0])

train_ds = GlandDataset(df_train, feature_cols, transform=train_tf)
val_ds   = GlandDataset(df_val,   feature_cols, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model_dl = GlandNetEffB0(n_features=len(feature_cols), fuse_morph=True, pretrained=True)
model_dl = model_dl.to(device)

optimizer = torch.optim.Adam(model_dl.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

def eval_epoch(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for imgs, feats, labels in loader:
            imgs   = imgs.to(device)
            feats  = feats.to(device)
            labels = labels.to(device)

            logits = model(imgs, feats)
            probs  = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    preds = (all_probs >= 0.5).astype(int)

    acc = accuracy_score(all_labels, preds)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = math.nan

    f1_maligno = f1_score(all_labels, preds, pos_label=1)

    return acc, auc, f1_maligno

# Entrenamiento
n_epochs = 20
best_auc = 0.0
best_state = None

for epoch in range(1, n_epochs+1):
    model_dl.train()
    running_loss = 0.0

    for imgs, feats, labels in train_loader:
        imgs   = imgs.to(device)
        feats  = feats.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model_dl(imgs, feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    val_acc, val_auc, val_f1 = eval_epoch(model_dl, val_loader, device)

    print(f"[Epoch {epoch:02d}] loss={train_loss:.4f} | "
          f"Val Acc={val_acc:.4f} | Val AUC={val_auc:.4f} | Val F1(maligno)={val_f1:.4f}")

    # guardar mejor modelo por AUC
    if val_auc > best_auc:
        best_auc = val_auc
        best_state = model_dl.state_dict().copy()

# Restaurar mejor estado
if best_state is not None:
    model_dl.load_state_dict(best_state)
    print(f"\nMejor modelo restaurado (AUC={best_auc:.4f})")
